# Prepare Data Inputs 

Input parameters as requested:

OHLC Data (Open, High, Low, Close)
- No volume used
- Add Indicators:
  - EMA(5), EMA(10)
  - RSI
  - MACD Histogram
  - ADX
  - ATR
  - Bollinger Band Width
  - Candle Body/Wick Ratio
  - Previous 3 Candle Trend (+2 = 2 UP, -2 = 2 DOWN)

  Anticipated calculation:

  EMA(5) & EMA(10):
  Common Smoothning factor: 2
  Multiplier would be 0.3333 for EMA(5) and 0.1818 EMA(10) Formula [Common factor/(Number of periods(in days) + 1)]
  EMA = (closing price) * Multiplier + Previous EMA * (1 - Multiplier) 

In [1]:
# Importing raw data 

import pandas as pd

In [2]:
col_names = ['timestamp', 'open', 'high', 'low', 'close', 'volume']

In [4]:
raw_df = pd.read_excel("../DATA_FOREX/1.EURUSD/Recent Data - (2023 - Latest)/RECENT_DATA_FILE_DUMP_EURUSD_M1.xlsx", header = None)

In [5]:
df = raw_df[0].str.split(";", expand=True)
df.columns = col_names
df['timestamp'] = pd.to_datetime(df['timestamp'], format="%Y%m%d %H%M%S")

# Convert price columns to float
df[['open', 'high', 'low', 'close']] = df[['open', 'high', 'low', 'close']].astype(float)

clean_df = df.drop(columns=['volume'])

print(clean_df.head())
print(df.index.is_unique)
print(df.index.duplicated())

            timestamp     open     high      low    close
0 2023-01-01 17:04:00  1.06970  1.06974  1.06970  1.06970
1 2023-01-01 17:05:00  1.06973  1.06978  1.06970  1.06971
2 2023-01-01 17:06:00  1.06966  1.06966  1.06966  1.06966
3 2023-01-01 17:08:00  1.06970  1.06974  1.06970  1.06974
4 2023-01-01 17:10:00  1.06975  1.06980  1.06972  1.06972
True
[False False False ... False False False]


In [6]:
# Set timestamp as index (critical for resampling)
clean_df.set_index('timestamp', inplace=True)
clean_df.index = pd.to_datetime(clean_df.index)
print(clean_df.index.duplicated().sum())
# To see duplicate values:
print(clean_df.index[clean_df.index.duplicated()])
clean_df = clean_df[~clean_df.index.duplicated(keep='first')]

60
DatetimeIndex(['2023-10-29 19:00:00', '2023-10-29 19:01:00',
               '2023-10-29 19:02:00', '2023-10-29 19:03:00',
               '2023-10-29 19:04:00', '2023-10-29 19:05:00',
               '2023-10-29 19:06:00', '2023-10-29 19:07:00',
               '2023-10-29 19:08:00', '2023-10-29 19:09:00',
               '2023-10-29 19:10:00', '2023-10-29 19:11:00',
               '2023-10-29 19:12:00', '2023-10-29 19:13:00',
               '2023-10-29 19:14:00', '2023-10-29 19:15:00',
               '2023-10-29 19:16:00', '2023-10-29 19:17:00',
               '2023-10-29 19:18:00', '2023-10-29 19:19:00',
               '2023-10-29 19:20:00', '2023-10-29 19:21:00',
               '2023-10-29 19:22:00', '2023-10-29 19:23:00',
               '2023-10-29 19:24:00', '2023-10-29 19:25:00',
               '2023-10-29 19:26:00', '2023-10-29 19:27:00',
               '2023-10-29 19:28:00', '2023-10-29 19:29:00',
               '2023-10-29 19:30:00', '2023-10-29 19:31:00',
               '2023-

In [7]:
#Combine Text File generator
import os
text_file_directory = "../DATA_FOREX/1.EURUSD/Recent Data - (2023 - Latest)/Text Files/"

def get_dir_files(text_file_directory: str) -> list: 
    """
    Syntax: os.listdir(path)   
    Parameters: path (optional) :  path of the directory  
    Return Type: This method returns the list of all files and directories in the specified path. The return type of this method is list. 
    """
    all_files = os.listdir(text_file_directory)
    return all_files

def combine_text_files(all_files: list):
    print(f"{all_files}")
    with open(f'{text_file_directory}/combined_txt_file.txt', 'a') as file:
        for a_file in all_files:
            print(f"{a_file}")
            with open(f'{text_file_directory}/{a_file}', 'r') as temp_file:
                file.write(temp_file.read() + '\n')
    return f'{text_file_directory}/combined_txt_file.txt'

all_files = get_dir_files(text_file_directory)
combine_text_file_path = combine_text_files(all_files)

print(f"New file created: {combine_text_file_path}")

['combined_txt_file.txt', 'DAT_ASCII_EURUSD_M1_2023.txt', 'DAT_ASCII_EURUSD_M1_202501.txt', 'DAT_ASCII_EURUSD_M1_2024.txt', 'DAT_ASCII_EURUSD_M1_202503.txt', 'DAT_ASCII_EURUSD_M1_202502.txt', 'DAT_ASCII_EURUSD_M1_202504.txt']
combined_txt_file.txt
DAT_ASCII_EURUSD_M1_2023.txt
DAT_ASCII_EURUSD_M1_202501.txt
DAT_ASCII_EURUSD_M1_2024.txt
DAT_ASCII_EURUSD_M1_202503.txt
DAT_ASCII_EURUSD_M1_202502.txt
DAT_ASCII_EURUSD_M1_202504.txt
New file created: ../DATA_FOREX/1.EURUSD/Recent Data - (2023 - Latest)/Text Files//combined_txt_file.txt


In [8]:
#Handling gap data to either forward fill missing data or completely remove large gaps in data
import re

def parse_gap_report(file_path):
    gaps = []
    with open(file_path, 'r') as f:
        for line in f:
            match = re.match(r"Gap of (\d+)s found between (\d{14}) and (\d{14})\.", line)
            if match:
                duration = int(match.group(1))
                start = pd.to_datetime(match.group(2), format="%Y%m%d%H%M%S")
                end = pd.to_datetime(match.group(3), format="%Y%m%d%H%M%S")
                gaps.append({"start": start, "end": end, "duration_s": duration})
    return gaps

def handling_gaps(clean_df, combine_text_file_path):
    temp = []
    missing_values = pd.DataFrame(temp)
    if combine_text_file_path:
        gaps = parse_gap_report(combine_text_file_path)
        # Forward fill small gaps (<= 300s) at 1-minute level
        clean_df = clean_df.asfreq('1T', method='ffill')
        # Flag large gaps (> 300s)
        for gap in gaps:
            if gap['duration_s'] > 300:
                # Mark period as unreliable (e.g., set to NaN or flag)
                clean_df.loc[gap['start']:gap['end']] = None
                missing_values = pd.DataFrame([gap['start'], gap['end']])
    else:
        # Forward fill all gaps if no gap report
        clean_df = clean_df.asfreq('1T', method='ffill')    
    return clean_df, missing_values


In [9]:

clean_df, missing_values_df = handling_gaps(clean_df, combine_text_file_path)
df_1min = clean_df.dropna()
print("\n1-Minute Data:")
print(df_1min.head())

# RESAMPLING 3-minute OHLC
df_3min = clean_df.resample('3T').agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last'
}).dropna()

# RESAMPLING 5-minute OHLC
df_5min = clean_df.resample('5T').agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last'
}).dropna()

# Preview
print("3-Minute Data:")
print(df_3min.head())

print("\n5-Minute Data:")
print(df_5min.head())


/tmp/ipykernel_49461/2102547322.py:22: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  clean_df = clean_df.asfreq('1T', method='ffill')



1-Minute Data:
                        open     high      low    close
timestamp                                              
2023-01-01 17:04:00  1.06970  1.06974  1.06970  1.06970
2023-01-01 17:05:00  1.06973  1.06978  1.06970  1.06971
2023-01-01 17:06:00  1.06966  1.06966  1.06966  1.06966
2023-01-01 17:07:00  1.06966  1.06966  1.06966  1.06966
2023-01-01 17:08:00  1.06970  1.06974  1.06970  1.06974


/tmp/ipykernel_49461/1285649055.py:7: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_3min = clean_df.resample('3T').agg({
/tmp/ipykernel_49461/1285649055.py:15: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_5min = clean_df.resample('5T').agg({


3-Minute Data:
                        open     high      low    close
timestamp                                              
2023-01-01 17:03:00  1.06970  1.06978  1.06970  1.06971
2023-01-01 17:06:00  1.06966  1.06974  1.06966  1.06974
2023-01-01 17:09:00  1.06970  1.06980  1.06970  1.06972
2023-01-01 17:12:00  1.06975  1.07066  1.06899  1.06899
2023-01-01 17:15:00  1.06788  1.06788  1.06788  1.06788

5-Minute Data:
                        open     high      low    close
timestamp                                              
2023-01-01 17:00:00  1.06970  1.06974  1.06970  1.06970
2023-01-01 17:05:00  1.06973  1.06978  1.06966  1.06974
2023-01-01 17:10:00  1.06975  1.07066  1.06899  1.06899
2023-01-01 17:15:00  1.06788  1.06788  1.06788  1.06788
2023-01-01 17:40:00  1.06951  1.06956  1.06934  1.06956


In [10]:
print(missing_values_df.head(10))

                    0
0 2025-02-12 17:17:00
1 2025-02-12 17:23:18


## Labeling 1-min

In [17]:
import numpy as np

df_1min = df_1min.copy()  

prev_close = df_1min['close'].shift(1)
conditions = [
    df_1min['close'] > prev_close,
    df_1min['close'] < prev_close
]
choices = [1, 2]

df_1min.loc[:, 'label'] = np.select(conditions, choices, default=0)
df_1min.loc[df_1min.index[0], 'label'] = 0 

In [19]:
df_1min.head(10)

,open,high,low,close,label
timestamp,,,,,
2023-01-01 17:04:00,1.06970,1.06974,1.06970,1.06970,0
2023-01-01 17:05:00,1.06973,1.06978,1.06970,1.06971,1
2023-01-01 17:06:00,1.06966,1.06966,1.06966,1.06966,2
2023-01-01 17:07:00,1.06966,1.06966,1.06966,1.06966,0
2023-01-01 17:08:00,1.06970,1.06974,1.06970,1.06974,1
2023-01-01 17:09:00,1.06970,1.06974,1.06970,1.06974,0
2023-01-01 17:10:00,1.06975,1.06980,1.06972,1.06972,2
2023-01-01 17:11:00,1.06972,1.06972,1.06972,1.06972,0
2023-01-01 17:12:00,1.06975,1.06980,1.06975,1.06980,1


## Adding Indicators

In [22]:
from stockstats import wrap
import pandas as pd

df_1min = df_1min.copy()
df_1min = df_1min.reset_index()  # stockstats requires 'timestamp' as a column

# Wrap with stockstats
sdf = wrap(df_1min)

# Compute indicators
sdf['close_5_ema']     # EMA(5)
sdf['close_10_ema']    # EMA(10)
sdf['rsi_14']          # RSI(14)
sdf['macdh']           # MACD Histogram
sdf['adx']             # ADX
sdf['atr']             # ATR
sdf['boll_ub']         # Bollinger Upper Band
sdf['boll_lb']         # Bollinger Lower Band
sdf['boll_width'] = sdf['boll_ub'] - sdf['boll_lb']  # Bollinger Band Width

# Manually compute Candle Body/Wick Ratio
sdf['body'] = abs(sdf['close'] - sdf['open'])
sdf['wick'] = sdf['high'] - sdf['low']
sdf['body_wick_ratio'] = sdf['body'] / sdf['wick'].replace(0, 1e-9)

# Optional: drop intermediate body/wick columns if you don't want them
# sdf.drop(columns=['body', 'wick'], inplace=True)


final_df = sdf[[
    'timestamp', 'open', 'high', 'low', 'close',
    'close_5_ema', 'close_10_ema', 'rsi_14', 'macdh',
    'adx', 'atr', 'boll_width', 'body_wick_ratio', 'label'
]]



                  timestamp     open     high      low    close  close_5_ema  \
1159210 2025-04-18 16:54:00  1.13918  1.13927  1.13918  1.13922     1.139181   
1159211 2025-04-18 16:55:00  1.13923  1.13923  1.13922  1.13923     1.139197   
1159212 2025-04-18 16:56:00  1.13923  1.13923  1.13922  1.13923     1.139208   
1159213 2025-04-18 16:57:00  1.13920  1.13927  1.13899  1.13923     1.139215   
1159214 2025-04-18 16:58:00  1.13900  1.13900  1.13898  1.13898     1.139137   

         close_10_ema     rsi_14     macdh        adx       atr  boll_width  \
1159210      1.139168  53.722323  0.000007  13.934408  0.000119    0.000297   
1159211      1.139179  54.416433  0.000010  15.040954  0.000112    0.000275   
1159212      1.139188  54.416433  0.000011  15.831344  0.000104    0.000280   
1159213      1.139196  54.416433  0.000012  15.692249  0.000117    0.000273   
1159214      1.139157  37.060117 -0.000005  15.907448  0.000126    0.000315   

         body_wick_ratio  label  
1159210   

In [23]:
len(final_df)

1159215

## Model Training

In [ ]:
df = final_df.dropna().copy()


X = df.drop(['timestamp', 'label'], axis=1)
y = df['label']


from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [27]:
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, shuffle=False)

model = LGBMClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.021537 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3060
[LightGBM] [Info] Number of data points in the train set: 927371, number of used features: 12
[LightGBM] [Info] Start training from score -0.986398
[LightGBM] [Info] Start training from score -1.155477
[LightGBM] [Info] Start training from score -1.164191


/home/khantil/anaconda3/envs/tmp/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Accuracy: 0.7623909283437499
              precision    recall  f1-score   support

           0       0.93      0.91      0.92     78997
           1       0.66      0.73      0.69     77197
           2       0.70      0.65      0.67     75649

    accuracy                           0.76    231843
   macro avg       0.76      0.76      0.76    231843
weighted avg       0.77      0.76      0.76    231843

